In [28]:
from pathlib import Path

# ============================================================
# FIND THE INDIA EV RANGE FILE
# ============================================================

PROJECT_ROOT = Path("/Users/piyushchauhan/EV-Intelligence-Platform")
DATA_DIR = PROJECT_ROOT / "data"

print("Data folder:")
print(DATA_DIR)

print("\nData folder exists:", DATA_DIR.exists())

print("\nFiles in data folder:")

for file in DATA_DIR.iterdir():
    if file.is_file():
        print(" -", file.name)

# ============================================================
# FIND INDIA EV RANGE FILE
# ============================================================

files = [
    f for f in DATA_DIR.iterdir()
    if f.is_file() and "india_ev_range" in f.name.lower()
]

print("\n" + "=" * 70)
print("INDIA EV RANGE FILES FOUND")
print("=" * 70)

for f in files:
    print(" -", f.name)

if not files:
    raise FileNotFoundError(
        "No India EV range file was found in the data folder."
    )

# Use the real CSV if available
csv_files = [
    f for f in files
    if f.suffix.lower() == ".csv"
]

if not csv_files:
    raise FileNotFoundError(
        "India EV range file exists, but no .csv version was found."
    )

path = csv_files[0]

print("\nUsing file:")
print(path)

# ============================================================
# READ RAW FILE
# ============================================================

with open(
    path,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    lines = []

    for _ in range(5):
        try:
            lines.append(next(f))
        except StopIteration:
            break

print("\n" + "=" * 70)
print("RAW FILE CONTENT")
print("=" * 70)

for i, line in enumerate(lines, 1):

    print(f"\nLINE {i}:")
    print(repr(line[:1500]))

Data folder:
/Users/piyushchauhan/EV-Intelligence-Platform/data

Data folder exists: True

Files in data folder:
 - india_ev_range_real.csv
 - .DS_Store
 - pm_edrive_ev_sales.csv
 - indian_ev_specs.csv

INDIA EV RANGE FILES FOUND
 - india_ev_range_real.csv

Using file:
/Users/piyushchauhan/EV-Intelligence-Platform/data/india_ev_range_real.csv

RAW FILE CONTENT

LINE 1:
'india_ev_range_dataset\n'

LINE 2:
'make,model,variant,battery_capacity_kWh,battery_capacity_kWh_new,motor_power_kW,vehicle_weight_kg,drag_coefficient_Cd,frontal_area_m2,rolling_resistance_Cr,motor_efficiency_pct,ARAI_certified_range_km,avg_driving_speed_kmh,ambient_temperature_C,ac_usage,terrain_type,traffic_condition,driving_style,passengers,battery_age_years,tyre_pressure_deviation_psi,city_road_type,energy_consumption_kWh_per_km,real_world_range_km,price_lakh_INR\n'

LINE 3:
'Tata,Nexon EV,Long Range,35.3,40.5,105,1610,0.34,2.4,0.0122,91,465,27.0,45.2,1,0,2,0,1,4.3,-2,Delhi,0.2504,141.0,17.5\n'

LINE 4:
'Tata,Nexon 

In [29]:
# ============================================================
# FIX INDIA EV RANGE DATASET
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Users/piyushchauhan/EV-Intelligence-Platform")
DATA_DIR = PROJECT_ROOT / "data"

INPUT_FILE = DATA_DIR / "india_ev_range_real.csv"

print("Input file:")
print(INPUT_FILE)

print("Exists:", INPUT_FILE.exists())


# ============================================================
# CORRECT COLUMN NAMES
# ============================================================

columns = [
    "make",
    "model",
    "variant",
    "battery_capacity_kWh",
    "battery_capacity_kWh_new",
    "motor_power_kW",
    "vehicle_weight_kg",
    "drag_coefficient_Cd",
    "frontal_area_m2",
    "rolling_resistance_Cr",
    "motor_efficiency_pct",
    "ARAI_certified_range_km",
    "avg_driving_speed_kmh",
    "ambient_temperature_C",
    "ac_usage",
    "terrain_type",
    "traffic_condition",
    "driving_style",
    "passengers",
    "battery_age_years",
    "tyre_pressure_deviation_psi",
    "city",
    "road_type",
    "energy_consumption_kWh_per_km",
    "real_world_range_km",
    "price_lakh_INR"
]

print("\nNumber of columns expected:", len(columns))


# ============================================================
# LOAD DATA
# ============================================================

df_ev_range = pd.read_csv(
    INPUT_FILE,
    skiprows=1,
    header=None,
    names=columns
)


# ============================================================
# BASIC CLEANING
# ============================================================

# Remove completely empty rows
df_ev_range = df_ev_range.dropna(how="all")

# Clean text columns
text_columns = [
    "make",
    "model",
    "variant",
    "terrain_type",
    "traffic_condition",
    "driving_style",
    "city",
    "road_type"
]

for col in text_columns:
    if col in df_ev_range.columns:
        df_ev_range[col] = (
            df_ev_range[col]
            .astype(str)
            .str.strip()
        )


# ============================================================
# CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "battery_capacity_kWh",
    "battery_capacity_kWh_new",
    "motor_power_kW",
    "vehicle_weight_kg",
    "drag_coefficient_Cd",
    "frontal_area_m2",
    "rolling_resistance_Cr",
    "motor_efficiency_pct",
    "ARAI_certified_range_km",
    "avg_driving_speed_kmh",
    "ambient_temperature_C",
    "ac_usage",
    "terrain_type",
    "traffic_condition",
    "driving_style",
    "passengers",
    "battery_age_years",
    "tyre_pressure_deviation_psi",
    "road_type",
    "energy_consumption_kWh_per_km",
    "real_world_range_km",
    "price_lakh_INR"
]

for col in numeric_columns:
    if col in df_ev_range.columns:
        df_ev_range[col] = pd.to_numeric(
            df_ev_range[col],
            errors="coerce"
        )


# ============================================================
# CHECK RESULT
# ============================================================

print("\n" + "=" * 70)
print("✅ INDIA EV DATASET FIXED")
print("=" * 70)

print("\nShape:")
print(df_ev_range.shape)

print("\nColumns:")
for i, col in enumerate(df_ev_range.columns, 1):
    print(f"{i}. {col}")

print("\nFirst 10 rows:")
display(df_ev_range.head(10))

print("\nData types:")
display(df_ev_range.dtypes)

print("\nMissing values:")
display(
    df_ev_range.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

Input file:
/Users/piyushchauhan/EV-Intelligence-Platform/data/india_ev_range_real.csv
Exists: True

Number of columns expected: 26

✅ INDIA EV DATASET FIXED

Shape:
(7292, 26)

Columns:
1. make
2. model
3. variant
4. battery_capacity_kWh
5. battery_capacity_kWh_new
6. motor_power_kW
7. vehicle_weight_kg
8. drag_coefficient_Cd
9. frontal_area_m2
10. rolling_resistance_Cr
11. motor_efficiency_pct
12. ARAI_certified_range_km
13. avg_driving_speed_kmh
14. ambient_temperature_C
15. ac_usage
16. terrain_type
17. traffic_condition
18. driving_style
19. passengers
20. battery_age_years
21. tyre_pressure_deviation_psi
22. city
23. road_type
24. energy_consumption_kWh_per_km
25. real_world_range_km
26. price_lakh_INR

First 10 rows:


,make,model,variant,battery_capacity_kWh,battery_capacity_kWh_new,motor_power_kW,vehicle_weight_kg,drag_coefficient_Cd,frontal_area_m2,rolling_resistance_Cr,...,traffic_condition,driving_style,passengers,battery_age_years,tyre_pressure_deviation_psi,city,road_type,energy_consumption_kWh_per_km,real_world_range_km,price_lakh_INR
0,make,model,variant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,city_road_type,NaN,NaN,NaN,NaN
1,Tata,Nexon EV,Long Range,35.3,40.5,105.0,1610.0,0.34,2.4,0.0122,...,2.0,0.0,1.0,4.3,-2.0,Delhi,0.2504,141.00,17.5,NaN
2,Tata,Nexon EV,Long Range,40.5,40.5,105.0,1610.0,0.34,2.4,0.0120,...,2.0,1.0,3.0,0.0,0.0,Delhi,0.1198,338.00,17.5,NaN
3,Tata,Nexon EV,Long Range,39.3,40.5,105.0,1610.0,0.34,2.4,0.0122,...,1.0,1.0,3.0,1.0,-2.0,Delhi,0.1254,313.40,17.5,NaN
4,Tata,Nexon EV,Long Range,39.0,40.5,105.0,1610.0,0.34,2.4,0.0120,...,0.0,1.0,1.0,1.2,0.0,Delhi,0.0883,441.75,17.5,NaN
5,Tata,Nexon EV,Long Range,37.9,40.5,105.0,1610.0,0.34,2.4,0.0120,...,2.0,0.0,3.0,2.1,0.0,Delhi,0.1960,193.40,17.5,NaN
6,Tata,Nexon EV,Long Range,38.8,40.5,105.0,1610.0,0.34,2.4,0.0126,...,0.0,1.0,2.0,1.4,-5.0,Delhi,0.1384,280.40,17.5,NaN
7,Tata,Nexon EV,Long Range,37.9,40.5,105.0,1610.0,0.34,2.4,0.0120,...,2.0,0.0,1.0,2.1,0.0,Delhi,0.2658,142.60,17.5,NaN
8,Tata,Nexon EV,Long Range,38.4,40.5,105.0,1610.0,0.34,2.4,0.0120,...,0.0,2.0,3.0,1.7,0.0,Delhi,0.1164,329.80,17.5,NaN
9,Tata,Nexon EV,Long Range,34.5,40.5,105.0,1610.0,0.34,2.4,0.0126,...,2.0,1.0,2.0,4.9,-5.0,Delhi,0.1774,194.50,17.5,NaN



Data types:


make                                 str
model                                str
variant                              str
battery_capacity_kWh             float64
battery_capacity_kWh_new         float64
motor_power_kW                   float64
vehicle_weight_kg                float64
drag_coefficient_Cd              float64
frontal_area_m2                  float64
rolling_resistance_Cr            float64
motor_efficiency_pct             float64
ARAI_certified_range_km          float64
avg_driving_speed_kmh            float64
ambient_temperature_C            float64
ac_usage                         float64
terrain_type                     float64
traffic_condition                float64
driving_style                    float64
passengers                       float64
battery_age_years                float64
tyre_pressure_deviation_psi      float64
city                                 str
road_type                        float64
energy_consumption_kWh_per_km    float64
real_world_range


Missing values:


,Missing Values
price_lakh_INR,7292
avg_driving_speed_kmh,1
real_world_range_km,1
energy_consumption_kWh_per_km,1
road_type,1
tyre_pressure_deviation_psi,1
battery_age_years,1
passengers,1
driving_style,1
traffic_condition,1


In [30]:
# ============================================================
# SAVE CLEANED INDIA EV DATASET
# ============================================================

OUTPUT_FILE = DATA_DIR / "india_ev_range_clean.csv"

df_ev_range.to_csv(
    OUTPUT_FILE,
    index=False
)

print("✅ Clean dataset saved!")
print("File:", OUTPUT_FILE)
print("Rows:", len(df_ev_range))
print("Columns:", len(df_ev_range.columns))

✅ Clean dataset saved!
File: /Users/piyushchauhan/EV-Intelligence-Platform/data/india_ev_range_clean.csv
Rows: 7292
Columns: 26
